In [ ]:
# 7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA

In [ ]:
from telegram import Update
from telegram.ext import Application, MessageHandler, filters, CallbackContext
import nest_asyncio
import asyncio

# Use nest_asyncio to allow nested event loops in Jupyter Notebook
nest_asyncio.apply()

# Replace with your bot token
BOT_TOKEN = '7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA'

async def get_chat_id(update: Update, context: CallbackContext) -> None:
    # Check if the update has a message
    if update.message:
        chat_id = update.effective_chat.id
        print(f"Group Chat ID: {chat_id}")
        await update.message.reply_text(f"Group Chat ID: {chat_id}")

async def run_bot():
    # Create the Application and pass the bot's token
    app = Application.builder().token(BOT_TOKEN).build()

    # Message handler to respond with the chat ID when a message is sent in a group
    app.add_handler(MessageHandler(filters.TEXT & filters.ChatType.GROUPS, get_chat_id))

    # Run the bot until the notebook is interrupted
    await app.run_polling()

# Run the bot
await run_bot()


In [ ]:
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters, CallbackContext
import nest_asyncio
import asyncio
import hashlib
import hmac
import requests
import time
import logging
import uuid

# Use nest_asyncio to allow nested event loops in Jupyter Notebook
nest_asyncio.apply()

# Replace with your bot token
BOT_TOKEN = '7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA'

# Sumsub API credentials
SUMSUB_APP_TOKEN = 'prd:Fhy56fEOaBdw28xHyEbLYmX8.9jhVeXYpbDnwZyOv3MximW4RlQwbPgZ8'
SUMSUB_SECRET_KEY = 'laNtfcWcj8k1J3goskEqwnA2T6p21ydD'
SUMSUB_BASE_URL = "https://api.sumsub.com"
REQUEST_TIMEOUT = 60

# Start Command Handler
async def start(update: Update, context: CallbackContext) -> None:
    await update.message.reply_text(
        "Welcome! You can use '@sumsub_pre_bot kyc' for KYC, '@sumsub_pre_bot poa' for POA, or '@sumsub_pre_bot full_kyc' for Full KYC."
    )

# Handle different commands
async def handle_message(update: Update, context: CallbackContext) -> None:
    message_text = update.message.text.lower()

    if '@sumsub_pre_bot kyc' in message_text:
        # Generate KYC link
        websdk_link = generate_websdk_link('basic-kyc-level', str(uuid.uuid4()))
        if websdk_link:
            await update.message.reply_text(f"Here is your KYC link: {websdk_link}")
        else:
            await update.message.reply_text("Sorry, there was an error generating your KYC link.")

    elif '@sumsub_pre_bot full_kyc' in message_text:
        # Generate Full KYC link
        websdk_link = generate_websdk_link('full-kyc-level', str(uuid.uuid4()))
        if websdk_link:
            await update.message.reply_text(f"Here is your Full KYC link: {websdk_link}")
        else:
            await update.message.reply_text("Sorry, there was an error generating your Full KYC link.")

    elif '@sumsub_pre_bot poa' in message_text:
        # Generate POA link
        websdk_link = generate_websdk_link('POA', str(uuid.uuid4()))
        if websdk_link:
            await update.message.reply_text(f"Here is your POA link: {websdk_link}")
        else:
            await update.message.reply_text("Sorry, there was an error generating your POA link.")

    # If the message contains @sumsub_pre_bot but is not a recognized command
    elif '@sumsub_pre_bot' in message_text:
        await update.message.reply_text(
            "I'm sorry, I don't understand that command. Please use '@sumsub_pre_bot kyc', '@sumsub_pre_bot poa', or '@sumsub_pre_bot full_kyc'."
        )

# Generate WebSDK Link
def generate_websdk_link(level_name, external_user_id, ttl_in_secs=1800, lang='en'):
    url = f"{SUMSUB_BASE_URL}/resources/sdkIntegrations/levels/{level_name}/websdkLink"
    params = {
        'ttlInSecs': ttl_in_secs,
        'externalUserId': external_user_id,
        'lang': lang
    }
    headers = {
        'Accept': 'application/json',
        'Content-Type': 'application/json'
    }

    try:
        response = sign_request(requests.Request('POST', url, params=params, headers=headers, data="{}"))
        session = requests.Session()
        resp = session.send(response, timeout=REQUEST_TIMEOUT)
        logging.info(f"Generate WebSDK Link Response: {resp.status_code} - {resp.text}")
        resp.raise_for_status()  # Raise an error for bad status codes
        websdk_url = resp.json().get('url')
        return websdk_url

    except requests.exceptions.RequestException as e:
        logging.error(f"Error generating WebSDK link: {e}")
        return None

# Sign API Request
def sign_request(request: requests.Request) -> requests.PreparedRequest:
    prepared_request = request.prepare()
    now = int(time.time())
    method = request.method.upper()
    path_url = prepared_request.path_url
    body = b'' if prepared_request.body is None else prepared_request.body
    if isinstance(body, str):
        body = body.encode('utf-8')

    # Create data to sign
    data_to_sign = str(now).encode('utf-8') + method.encode('utf-8') + path_url.encode('utf-8') + body
    signature = hmac.new(
        SUMSUB_SECRET_KEY.encode('utf-8'),
        data_to_sign,
        digestmod=hashlib.sha256
    )

    prepared_request.headers['X-App-Token'] = SUMSUB_APP_TOKEN
    prepared_request.headers['X-App-Access-Ts'] = str(now)
    prepared_request.headers['X-App-Access-Sig'] = signature.hexdigest()

    logging.debug(f"Signed Request Headers: {prepared_request.headers}")
    logging.debug(f"Signed Request URL: {prepared_request.url}")
    return prepared_request

# Run the bot
async def run_bot():
    logging.basicConfig(level=logging.INFO)

    # Create the Application and pass the bot's token
    app = Application.builder().token(BOT_TOKEN).build()

    # Command handler for /start
    app.add_handler(CommandHandler("start", start))

    # Message handler to respond when the bot is mentioned with "kyc", "poa", or "full_kyc"
    app.add_handler(MessageHandler(filters.MENTIONED & filters.TEXT, handle_message))

    # Run the bot until interrupted
    await app.run_polling()

# Run the bot
await run_bot()


In [ ]:
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters, CallbackContext
import nest_asyncio
import asyncio
import hashlib
import hmac
import requests
import time
import logging
import uuid
import sqlite3

# Use nest_asyncio to allow nested event loops in Jupyter Notebook
nest_asyncio.apply()

# Replace with your bot token
BOT_TOKEN = '7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA'

# Sumsub API credentials
SUMSUB_APP_TOKEN = 'prd:Fhy56fEOaBdw28xHyEbLYmX8.9jhVeXYpbDnwZyOv3MximW4RlQwbPgZ8'
SUMSUB_SECRET_KEY = 'laNtfcWcj8k1J3goskEqwnA2T6p21ydD'
SUMSUB_BASE_URL = "https://api.sumsub.com"
REQUEST_TIMEOUT = 60

# Initialize SQLite database
def init_db():
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS users (
            user_id INTEGER PRIMARY KEY,
            total_credits INTEGER DEFAULT 1      
        )
    ''')
    conn.commit()
    conn.close()

# Add a new user if they don't exist in the database
def add_user(user_id):
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    cursor.execute('''
        INSERT OR IGNORE INTO users (user_id) VALUES (?)
    ''', (user_id,))
    conn.commit()
    conn.close()

# Function to check total credits
def check_credits(user_id):
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    cursor.execute("SELECT total_credits FROM users WHERE user_id=?", (user_id,))
    result = cursor.fetchone()
    conn.close()
    return result[0] if result else 0

# Deduct a credit from the user
def deduct_credit(user_id):
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    cursor.execute("UPDATE users SET total_credits = total_credits - 1 WHERE user_id = ?", (user_id,))
    conn.commit()
    conn.close()

# Start command handler
async def start(update: Update, context: CallbackContext) -> None:
    user_id = update.message.from_user.id
    add_user(user_id)  # Add user if they don't exist
    await update.message.reply_text("Welcome! You have 1 credit, which you can spend on either KYC, Full KYC, or POA.")

# Handle KYC, Full KYC, and POA requests
async def handle_message(update: Update, context: CallbackContext) -> None:
    user_id = update.message.from_user.id
    message_text = update.message.text.lower()

    add_user(user_id)  # Ensure user is in the database

    # Check total credits
    total_credits = check_credits(user_id)
    if total_credits > 0:
        if '@sumsub_pre_bot kyc' in message_text:
            websdk_link = generate_websdk_link('basic-kyc-level', str(uuid.uuid4()))
            if websdk_link:
                await update.message.reply_text(f"Here is your KYC link: {websdk_link}")
                deduct_credit(user_id)  # Deduct one credit
            else:
                await update.message.reply_text("Sorry, there was an error generating your KYC link.")

        elif '@sumsub_pre_bot full_kyc' in message_text:
            websdk_link = generate_websdk_link('full-kyc-level', str(uuid.uuid4()))
            if websdk_link:
                await update.message.reply_text(f"Here is your Full KYC link: {websdk_link}")
                deduct_credit(user_id)  # Deduct one credit
            else:
                await update.message.reply_text("Sorry, there was an error generating your Full KYC link.")

        elif '@sumsub_pre_bot poa' in message_text:
            websdk_link = generate_websdk_link('poa', str(uuid.uuid4()))
            if websdk_link:
                await update.message.reply_text(f"Here is your POA link: {websdk_link}")
                deduct_credit(user_id)  # Deduct one credit
            else:
                await update.message.reply_text("Sorry, there was an error generating your POA link.")
        else:
            await update.message.reply_text("Invalid command. Please use '@sumsub_pre_bot kyc', '@sumsub_pre_bot full_kyc', or '@sumsub_pre_bot poa'.")
    else:
        await update.message.reply_text("You have no credits left.")

# Generate WebSDK link
def generate_websdk_link(level_name, external_user_id, ttl_in_secs=1800, lang='en'):
    url = f"{SUMSUB_BASE_URL}/resources/sdkIntegrations/levels/{level_name}/websdkLink"
    params = {
        'ttlInSecs': ttl_in_secs,
        'externalUserId': external_user_id,
        'lang': lang
    }
    headers = {
        'Accept': 'application/json',
        'Content-Type': 'application/json'
    }
    try:
        response = sign_request(requests.Request('POST', url, params=params, headers=headers, data="{}"))
        session = requests.Session()
        resp = session.send(response, timeout=REQUEST_TIMEOUT)
        logging.info(f"Generate WebSDK Link Response: {resp.status_code} - {resp.text}")
        resp.raise_for_status()
        websdk_url = resp.json().get('url')
        return websdk_url
    except Exception as e:
        logging.error(f"Error generating WebSDK link: {e}")
        return None

# Sign API request
def sign_request(request: requests.Request) -> requests.PreparedRequest:
    prepared_request = request.prepare()
    now = int(time.time())
    method = request.method.upper()
    path_url = prepared_request.path_url
    body = b'' if prepared_request.body is None else prepared_request.body
    if isinstance(body, str):
        body = body.encode('utf-8')
    data_to_sign = str(now).encode('utf-8') + method.encode('utf-8') + path_url.encode('utf-8') + body
    signature = hmac.new(
        SUMSUB_SECRET_KEY.encode('utf-8'),
        data_to_sign,
        digestmod=hashlib.sha256
    )
    prepared_request.headers['X-App-Token'] = SUMSUB_APP_TOKEN
    prepared_request.headers['X-App-Access-Ts'] = str(now)
    prepared_request.headers['X-App-Access-Sig'] = signature.hexdigest()

    logging.debug(f"Signed Request Headers: {prepared_request.headers}")
    logging.debug(f"Signed Request URL: {prepared_request.url}")
    return prepared_request

# Run the bot
async def run_bot():
    # Set up logging
    logging.basicConfig(level=logging.INFO)

    # Create the Application and pass the bot's token
    app = Application.builder().token(BOT_TOKEN).build()

    # Initialize the SQLite database
    init_db()

    # Command handler for /start
    app.add_handler(CommandHandler("start", start))

    # Message handler to respond when the bot is mentioned with "kyc", "full_kyc", or "poa"
    app.add_handler(MessageHandler(filters.TEXT & filters.ChatType.GROUPS, handle_message))

    # Run the bot until the notebook is interrupted
    await app.run_polling()

# Run the bot
await run_bot()


In [1]:
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters, CallbackContext
import nest_asyncio
import asyncio
import hashlib
import hmac
import requests
import time
import logging
import uuid
import sqlite3

# Use nest_asyncio to allow nested event loops in Jupyter Notebook
nest_asyncio.apply()

BOT_TOKEN = '7580665549:AAEiILYjLzZg34wIFOBZB-FtfUhsjQMBUrA'

# Sumsub API credentials
SUMSUB_APP_TOKEN = 'prd:Fhy56fEOaBdw28xHyEbLYmX8.9jhVeXYpbDnwZyOv3MximW4RlQwbPgZ8'
SUMSUB_SECRET_KEY = 'laNtfcWcj8k1J3goskEqwnA2T6p21ydD'
SUMSUB_BASE_URL = "https://api.sumsub.com"
REQUEST_TIMEOUT = 60

# Initialize SQLite database
def init_db():
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS users (
            user_id INTEGER PRIMARY KEY,
            total_credits INTEGER DEFAULT 2      
        )
    ''')
    conn.commit()
    conn.close()

# Add a new user if they don't exist in the database
def add_user(user_id):
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    cursor.execute('''
        INSERT OR IGNORE INTO users (user_id) VALUES (?)
    ''', (user_id,))
    conn.commit()
    conn.close()

# Function to check total credits
def check_credits(user_id):
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    cursor.execute("SELECT total_credits FROM users WHERE user_id=?", (user_id,))
    result = cursor.fetchone()
    conn.close()
    return result[0] if result else 0

# Deduct a credit from the user
def deduct_credit(user_id):
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    cursor.execute("UPDATE users SET total_credits = total_credits - 1 WHERE user_id = ?", (user_id,))
    conn.commit()
    conn.close()


In [2]:
# Start command handler
async def start(update: Update, context: CallbackContext) -> None:
    user_id = update.message.from_user.id
    add_user(user_id)  # Add user if they don't exist
    await update.message.reply_text("Welcome! You have 1 credit, which you can spend on either KYC, Full KYC, or POA.")

# Handle KYC, Full KYC, and POA requests
async def handle_message(update: Update, context: CallbackContext) -> None:
    user_id = update.message.from_user.id
    message_text = update.message.text.lower()

    add_user(user_id)  # Ensure user is in the database

    # Check total credits
    total_credits = check_credits(user_id)
    if total_credits > 0:
        if '@sumsub_pre_bot kyc' in message_text:
            websdk_link = generate_websdk_link('basic-kyc-level', str(uuid.uuid4()))
            if websdk_link:
                await update.message.reply_text(f"Here is your KYC link: {websdk_link}")
                deduct_credit(user_id)  # Deduct one credit
            else:
                await update.message.reply_text("Sorry, there was an error generating your KYC link.")

        elif '@sumsub_pre_bot full_kyc' in message_text:
            websdk_link = generate_websdk_link('full-kyc-level', str(uuid.uuid4()))
            if websdk_link:
                await update.message.reply_text(f"Here is your Full KYC link: {websdk_link}")
                deduct_credit(user_id)  # Deduct one credit
            else:
                await update.message.reply_text("Sorry, there was an error generating your Full KYC link.")

        elif '@sumsub_pre_bot poa' in message_text:
            websdk_link = generate_websdk_link('poa', str(uuid.uuid4()))
            if websdk_link:
                await update.message.reply_text(f"Here is your POA link: {websdk_link}")
                deduct_credit(user_id)  # Deduct one credit
            else:
                await update.message.reply_text("Sorry, there was an error generating your POA link.")
        else:
            await update.message.reply_text("Invalid command. Please use '@sumsub_pre_bot kyc', '@sumsub_pre_bot full_kyc', or '@sumsub_pre_bot poa'.")
    else:
        await update.message.reply_text("You have no credits left.")


In [ ]:
# Generate WebSDK link
def generate_websdk_link(level_name, external_user_id, ttl_in_secs=1800, lang='en'):
    url = f"{SUMSUB_BASE_URL}/resources/sdkIntegrations/levels/{level_name}/websdkLink"
    params = {
        'ttlInSecs': ttl_in_secs,
        'externalUserId': external_user_id,
        'lang': lang
    }
    headers = {
        'Accept': 'application/json',
        'Content-Type': 'application/json'
    }
    try:
        response = sign_request(requests.Request('POST', url, params=params, headers=headers, data="{}"))
        session = requests.Session()
        resp = session.send(response, timeout=REQUEST_TIMEOUT)
        logging.info(f"Generate WebSDK Link Response: {resp.status_code} - {resp.text}")
        resp.raise_for_status()
        websdk_url = resp.json().get('url')
        return websdk_url
    except Exception as e:
        logging.error(f"Error generating WebSDK link: {e}")
        return None

# Sign API request
def sign_request(request: requests.Request) -> requests.PreparedRequest:
    prepared_request = request.prepare()
    now = int(time.time())
    method = request.method.upper()
    path_url = prepared_request.path_url
    body = b'' if prepared_request.body is None else prepared_request.body
    if isinstance(body, str):
        body = body.encode('utf-8')
    data_to_sign = str(now).encode('utf-8') + method.encode('utf-8') + path_url.encode('utf-8') + body
    signature = hmac.new(
        SUMSUB_SECRET_KEY.encode('utf-8'),
        data_to_sign,
        digestmod=hashlib.sha256
    )
    prepared_request.headers['X-App-Token'] = SUMSUB_APP_TOKEN
    prepared_request.headers['X-App-Access-Ts'] = str(now)
    prepared_request.headers['X-App-Access-Sig'] = signature.hexdigest()

    logging.debug(f"Signed Request Headers: {prepared_request.headers}")
    logging.debug(f"Signed Request URL: {prepared_request.url}")
    return prepared_request

# Run the bot
async def run_bot():
    # Set up logging
    logging.basicConfig(level=logging.INFO)

    # Create the Application and pass the bot's token
    app = Application.builder().token(BOT_TOKEN).build()

    # Initialize the SQLite database
    init_db()

    # Command handler for /start
    app.add_handler(CommandHandler("start", start))

    # Message handler to respond when the bot is mentioned with "kyc", "full_kyc", or "poa"
    app.add_handler(MessageHandler(filters.TEXT & filters.ChatType.GROUPS, handle_message))

    # Run the bot until the notebook is interrupted
    await app.run_polling()

# Run the bot
await run_bot()
